In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score
)

# Create output folders
os.makedirs('../outputs/plots', exist_ok=True)

# Load model and data
pipe = joblib.load('../outputs/models/lgbm_pipeline.pkl')
df = pd.read_csv('../data/processed/fused.csv')

# Prepare features
DROP = [
    c for c in
    ['target', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
    if c in df.columns
]

X = df.drop(columns=DROP)
y = df['target']

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# -------------------------
# Precision-Recall Curve
# -------------------------
y_prob = pipe.predict_proba(X_te)[:, 1]

prec, rec, thresh = precision_recall_curve(
    y_te,
    y_prob
)

ap = average_precision_score(
    y_te,
    y_prob
)

plt.figure(figsize=(8, 5))
plt.plot(
    rec,
    prec,
    linewidth=1.5,
    label=f'AP = {ap:.3f}'
)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()

plt.savefig(
    '../outputs/plots/pr_curve.png',
    dpi=150
)

plt.close()

print('PR curve saved.')

# -------------------------
# Best Threshold
# -------------------------
f1s = []

for t in thresh:
    pred = (y_prob >= t).astype(int)
    f1 = f1_score(
        y_te,
        pred,
        average='macro'
    )
    f1s.append(f1)

best_idx = np.argmax(f1s)
best_thresh = thresh[best_idx]
best_f1 = f1s[best_idx]

print(
    f'Best threshold: {best_thresh:.3f} | '
    f'Best Macro F1: {best_f1:.4f}'
)

# -------------------------
# Noise Sensitivity Test
# -------------------------
noise_levels = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.30
]

results = []

numeric_cols = X_te.select_dtypes(
    include=np.number
).columns

for sigma in noise_levels:

    X_noisy = X_te.copy()

    noise = np.random.normal(
        0,
        sigma * X_te[numeric_cols].std(),
        X_te[numeric_cols].shape
    )

    X_noisy[numeric_cols] = (
        X_te[numeric_cols] + noise
    )

    y_prob_n = pipe.predict_proba(
        X_noisy
    )[:, 1]

    pred = (
        y_prob_n >= best_thresh
    ).astype(int)

    f1 = f1_score(
        y_te,
        pred,
        average='macro'
    )

    results.append({
        'noise_sigma': sigma,
        'macro_f1': round(f1, 4)
    })

    print(
        f'Noise σ={sigma:.2f}: '
        f'Macro F1 = {f1:.4f}'
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    '../outputs/noise_sensitivity.csv',
    index=False
)

print('Noise sensitivity saved.')

PR curve saved.
Best threshold: 0.996 | Best Macro F1: 1.0000
Noise σ=0.00: Macro F1 = 1.0000
Noise σ=0.05: Macro F1 = 0.9190
Noise σ=0.10: Macro F1 = 0.8826
Noise σ=0.15: Macro F1 = 0.8478
Noise σ=0.20: Macro F1 = 0.8071
Noise σ=0.30: Macro F1 = 0.7853
Noise sensitivity saved.
